[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/class-demos/session02_python_essentials.ipynb)

# Session 2 live demo — the Python that bites, in the order the deck shows it

**Session 2 · the in-class demo — runs on the free tier of Colab, nothing to install**

Open this before class and run it once. The deck carries 21 code slides; these
are the ones worth *running* in the room, because the output is the argument.
The rest of the slides read fine as text.

Every printed value below was captured from this file.

Each heading names the slide its cell accompanies; the outputs below were saved from a real run.

## Slides 6-7: mutability, and the aliasing trap

In [1]:
a = [1, 2, 3]
b = a                      # not a copy: another name for the same list
b.append(4)
print("aliasing:      ", a, b)
# aliasing:       [1, 2, 3, 4] [1, 2, 3, 4]

c = a[:]                   # a copy
c.append(5)
print("after copy:    ", a, c)
# after copy:     [1, 2, 3, 4] [1, 2, 3, 4, 5]

aliasing:       [1, 2, 3, 4] [1, 2, 3, 4]
after copy:     [1, 2, 3, 4] [1, 2, 3, 4, 5]


## Slide 11: the mutable default pitfall

In [2]:
# ruff flags the next line as B006, which is the slide's entire point: the
# linter catches this bug, and the room should see that it does.
def bad(item, log=[]):     # noqa: B006 — evaluated ONCE, at definition
    log.append(item)
    return log

print("mutable default:", bad("a"), bad("b"))
# mutable default: ['a', 'b'] ['a', 'b']    <- the second call sees the first

def good(item, log=None):
    log = [] if log is None else log
    log.append(item)
    return log

print("fixed:         ", good("a"), good("b"))
# fixed:          ['a'] ['b']

mutable default: ['a', 'b'] ['a', 'b']
fixed:          ['a'] ['b']


## Slide 13: LEGB, and why `global` is the wrong instinct

In [3]:
threshold = 0.5

def classify(score):
    threshold = 0.9        # local; shadows the global, does not change it
    return score > threshold

print("scope:         ", classify(0.7), threshold)
# scope:          False 0.5

scope:          False 0.5


## Slides 14-16: comprehensions, and when not to

In [4]:
scores = [0.91, 0.55, 0.87, 0.42, 0.95]
print("comprehension: ", [round(s * 100) for s in scores if s > 0.6])
# comprehension:  [91, 87, 95]

comprehension:  [91, 87, 95]


## Slides 17-19: generators are lazy, and that is the point

In [5]:
def running_max(values):
    best = float("-inf")
    for v in values:
        best = max(best, v)
        yield best

print("generator:     ", list(running_max(scores)))
# generator:      [0.91, 0.91, 0.91, 0.91, 0.95]

generator:      [0.91, 0.91, 0.91, 0.91, 0.95]


## Slides 21-25: the class shape every sklearn estimator has

In [6]:
class Standardizer:
    def fit(self, values):
        self.mean_ = sum(values) / len(values)          # trailing _ = learned
        spread = (sum((v - self.mean_) ** 2 for v in values) / len(values)) ** 0.5
        self.scale_ = spread or 1.0
        return self                                     # fit returns self

    def transform(self, values):
        return [(v - self.mean_) / self.scale_ for v in values]

    def __repr__(self):                                 # dunder: how print sees it
        return f"Standardizer(mean_={getattr(self, 'mean_', None)})"

scaler = Standardizer().fit(scores)
print("estimator:     ", scaler, [round(v, 2) for v in scaler.transform(scores)])
# estimator:      Standardizer(mean_=0.74) [0.8, -0.89, 0.61, -1.5, 0.98]

estimator:      Standardizer(mean_=0.74) [0.8, -0.89, 0.61, -1.5, 0.98]


## Slides 28-29: exceptions, and EAFP over LBYL

In [7]:
config = {"alpha": "0.05"}

if "alpha" in config and isinstance(config["alpha"], float):   # LBYL
    lbyl = config["alpha"]
else:
    lbyl = 0.1

try:                                                            # EAFP
    eafp = float(config["alpha"])
except (KeyError, TypeError, ValueError):
    eafp = 0.1

print("LBYL vs EAFP:  ", lbyl, eafp)
# LBYL vs EAFP:   0.1 0.05     <- the check was checking the wrong thing

LBYL vs EAFP:   0.1 0.05


## Slides 31-32: the idioms they will use every week

In [8]:
from pathlib import Path

names = ["ridge", "lasso", "tree"]
for rank, (name, score) in enumerate(zip(names, scores), start=1):
    print(f"  {rank}. {name:<6} {score:.2f}")
#   1. ridge  0.91
#   2. lasso  0.55
#   3. tree   0.87

results = Path("experiments") / "run-01" / "metrics.csv"
print("pathlib:       ", results.name, results.suffix, results.parent)
# pathlib:        metrics.csv .csv experiments/run-01

  1. ridge  0.91
  2. lasso  0.55
  3. tree   0.87
pathlib:        metrics.csv .csv experiments/run-01
